# 5-1: Topic Modeling in Python

In this tutorial, we'll learn a basic topic-modeling workflow using a small corpus of United Nations human-rights reports. The goal is to undCtopic models turn a set of documents into patterns of word co-occurrence, and how to interpret those patterns carefully.

Our workflow will look like this:

1. Load a folder of text files into a dataframe
2. Preprocess the texts into cleaner tokens
3. Build a document-term matrix
4. Fit an LDA topic model with `scikit-learn`
5. Inspect topic words and document-topic mixtures
6. Visualize the model with `pyLDAvis`
7. Compare the same basic workflow in `gensim`

By the end, you should be able to explain what a document-term matrix is, what LDA means by a topic, why preprocessing choices matter, and how to begin interpreting a topic model as an exploratory humanities method.

## Learning Objectives

1. Load multiple plain-text files from a local data folder
2. Preprocess text by tokenizing, lowercasing, and removing stop words
3. Create a document-term matrix with `CountVectorizer`
4. Fit and inspect an LDA topic model with `scikit-learn`
5. Visualize and interpret topic-model output with `pyLDAvis`
6. Run a comparable topic model with `gensim`

This tutorial also marks our first dedicated foray into __machine learning__. Machine learning is a family of computational methods in which a model learns patterns from data and uses those patterns to make predictions, group examples, or represent information. Instead of following only hand-written rules, a machine-learning model adjusts itself based on the examples it is given. In this way, it's said to be _learning_, but of course, in a strictly limited sense.

There are a few categories of machine learning. They are:

- __Supervised learning__: the model learns from examples that already have labels, such as texts marked as `positive` or `negative`, and then tries to predict labels for new examples.
- __Unsupervised learning__: the model looks for patterns in data without pre-existing labels. Topic modeling fits here because LDA looks for clusters of words across documents without being told the topics in advance.
- __Self-supervised learning__: the model creates a training task from the data itself, such as learning word relationships by predicting nearby or missing words. Many word-embedding and LLM methods use this kind of learning.
- __Reinforcement learning__: the model learns by trying actions and receiving rewards or penalties. This is less central to our text-analysis workflow, but it is another major category of machine learning.

As we work through this week's tutorials, we'll discuss precisely where and how they fit within the umbrella of machine learning. The whole area of research has become extremely relevant to DH scholarship. Many machine learning methods are used in the field to navigate information and sources, retrieve information, classify texts, test hypotheses, and much more.

## Topic Modeling: An Overview

Before diving into the computational approach, let's first consider __topics__. What is a topic, actually? How are topics identified in texts?

There are a lot of ways to answer those questions. But one way to think of topics and how they're signaled in texts is by vocabulary/word choice. That is, certain words appearing in a text are indicative of certain topics. If a text has a high frequency of words related to cats, dogs, animal training, animal health, and so forth, it might be indicative of the topic "pets."

But it's not just word frequency that indicates topics––it's also the groupings of specific words. A text might have high frequencies of words related to warfare, but unless it also has groupings of words related to places, dates, historic figures, or specific weapons and technology, it might be difficult to tell which war is the topic in the text.

__Topic modeling__ is one way to try to identify textual topics using computational methods. A __topic model__ is an unsupervised machine-learning method for finding clusters of words that tend to appear together across documents. In __Latent Dirichlet Allocation__, usually shortened to __LDA__, a topic is represented as a weighted list of words, and a document is represented as a mixture of topics.

That last sentence is important. LDA doesn't read a document the way a person reads it. It doesn't know what a theme, argument, genre, or historical context is. It only looks for patterns in word co-occurrence. The model asks: which words tend to cluster together, and which documents contain those clusters?

Because of that, topic modeling is most useful as an exploratory method. It can help us notice patterns that we might want to investigate more closely, but the model output still needs human interpretation. Quite often, it will identify word clusters (i.e., topics) that surprise us, or don't actually have much interpretable meaning. Other times, it may validate what we already know or believe about our texts. Quite often, it might also raise questions about how or why certain word clusters or topics appear in texts. It will never tell you why or how they appear––just that they're there––requiring further investigation. That's why topic modeling is typically used as an exploratory method.

### Important Interpretive Caution

Topic models are not neutral summaries of texts. They are measurements of a particular corpus after a particular preprocessing workflow.

If we remove different words, choose a different number of topics, or use a different set of documents, we'll get different topics. This makes topic modeling difficult to replicate. It also means that the method is interpretive from the beginning, requiring researchers to define topic thresholds, define documents, etc.

As you work through this notebook, keep asking:

- What counts as a document in this corpus?
- Which words did we remove before modeling?
- Which repeated boilerplate language might still shape the results?
- Do the topics help us ask better questions about the texts?
- Which claims would require us to go back and read the documents themselves?

## The Human Rights Corpus

We'll use the text files in `../data/human-rights`. These are eleven United Nations Human Rights Council Universal Periodic Review reports from 2013 and 2014.

This is a good corpus for topic modeling demos because the documents have similar institutional form, but they discuss different countries. That also creates a challenge: because the documents share lots of official language, a topic model can easily rediscover the boilerplate instead of more meaningful differences among the reports. Our preprocessing choices will try to reduce some of that shared institutional vocabulary. Then perhaps we can identify topics unique to individual reports or those that are surprisingly universal.

## Setup

We'll use these libraries:

- `glob` and `os` for finding and opening files
- `pandas` for tabular data
- `gensim` for text preprocessing and a second topic model
- `scikit-learn` for the main LDA model
- `matplotlib` and `seaborn` for plots
- `pyLDAvis` for the interactive topic-model visualization

`pyLDAvis` is not listed in the course `environment.yml`, so we'll install it in the next cell. If it's already installed, the command should simply tell you that the requirement is already satisfied.

In [ ]:
%pip install pyLDAvis

There are some new libraries and modules in this tutorial. Can you look them up and find their documentation? Check out the resources section of our course repository for more info. Or, ask an LLM about these libraries and modules. What are they used for? What are their purposes?

In [ ]:
%matplotlib inline

from collections import Counter
from glob import glob
import os
import re

import gensim
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gensim import corpora
from gensim.models.ldamodel import LdaModel
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

import pyLDAvis

## Step 1: Load The Text Files

Our documents are stored as separate `.txt` files. To make them easier to analyze, we'll read each file and store the results in one dataframe.

Each row in the dataframe will represent one document. The `file` column stores the filename, and the `text` column stores the full text of that report.

First let's get the path to the files all situated:

In [ ]:
data_folder = "../data/human-rights"
file_paths = sorted(glob(os.path.join(data_folder, "*.txt")))

print(f"Text files found: {len(file_paths)}")
file_paths[:3]


Now we'll loop through those file paths, open each text file, and collect the results in a list of dictionaries. Then `pd.DataFrame()` turns that list into a table.


In [ ]:
records = []

for file_path in file_paths:
    file_name = os.path.basename(file_path)

    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()

    records.append({
        "file": file_name,
        "document": os.path.splitext(file_name)[0],
        "text": text,
    })

human_rights = pd.DataFrame(records)
human_rights["word_count"] = human_rights["text"].str.split().str.len()

human_rights

Notice the word counts here. These documents are fairly uniform and long enough to contain meaningful topics. That's something you need to consider when topic modeling. __Documents__ are a variable in the model. They need to have enough words to contain potential topics for identification. Longer documents may have different topics or more topics than shorter documents. Novel-length documents are too long to compare against article-length documents. Short poems or social media posts might not be long enough to contain LDA topics. Sometimes, you'll need to split longer documents apart to compare their topics effectively––think chapters of novels and topic comparison, or sections of longer texts to compare against each other. In any case, though, you should assess document length any time you're topic modeling. Documents should be relatively comparable in terms of word count in order to make meaningful comparisons between topics.

Anyway, let's preview one document to make sure the texts loaded correctly. Notice the official language at the beginning of the report. That kind of repeated institutional language will matter when we preprocess the corpus.

In [ ]:
print(human_rights.loc[0, "text"][:1000])


## Step 2: Preprocess The Text

Before we build a topic model, we'll need to do some preprocessing.

In this case, our preprocessing will do four things:

1. Lowercase the text
2. Remove punctuation and accent marks
3. Keep words with at least three characters
4. Remove stop words

Stop words matter a lot for topic modeling. LDA looks for words that help explain patterns across documents. If very common words like `the` and `and` remain in the corpus, they can become part of the model even though they tell us very little about the reports. The same problem happens with corpus-specific boilerplate. In these UN reports, words such as `united`, `nations`, `council`, `review`, and `recommendations` appear often because of the document format, not because they identify a meaningful topic.

So we'll build stop words in two stages. First, we'll use `scikit-learn`'s built-in English stop words. Then we'll add a small set of corpus-specific stop words based on the language we see in these Human Rights reports.

In [ ]:
raw_tokens = []

for text in human_rights["text"]:
    text = text.lower()
    # notice this regex pattern
    # only keeps alphabetic characters with a minimum length of 3
    # ignores numeric characters
    tokens = re.findall(r"[a-z]{3,}", text)
    # Adds tokens to the full corpus-wide token list
    raw_tokens.extend(tokens)

raw_word_counts = Counter(raw_tokens)

raw_top_words = pd.DataFrame(
    raw_word_counts.most_common(30),
    columns=["word", "frequency"]
)

raw_top_words

This quick frequency table is not the final answer, but it helps us make preprocessing decisions. Can we deduce why some of these words would be the most frequent in the corpus? Some high-frequency words are the usual stop words in English (articles, prepositions, etc.). Others are institutional words from the UN report format. 

If those words stay in the corpus, the topic model may mostly discover that these documents are UN review reports, which we already know. Others might help us understand the topics specific to individual documents, meaning we'd want to leave them for our model.

How do we decide which words stay or get removed in preprocessing? We shouldn't remove a word just because it is frequent. Words like `women`, `children`, `education`, or `violence` may be frequent _and_ meaningful for this corpus.

Here's an attempt at choosing the removable (i.e., _less meaningful_ in terms of their signal for topics) words. What do you think of this list?

In [ ]:
UN_stopwords = {
    "accepted",
    "also",
    "assembly",
    "council",
    "country",
    "delegation",
    "document",
    "documents",
    "general",
    "group",
    "human",
    "national",
    "nations",
    "noted",
    "paragraph",
    "paragraphs",
    "periodic",
    "recommendation",
    "recommendations",
    "recommended",
    "report",
    "reports",
    "review",
    "right",
    "rights",
    "session",
    "state",
    "states",
    "united",
    "universal",
    "working",
}

UN_stopword_counts = pd.DataFrame(
    [(word, raw_word_counts[word]) for word in sorted(UN_stopwords)],
    columns=["word", "raw_frequency"]
).sort_values("raw_frequency", ascending=False)

UN_stopword_counts.head(15)

The table above lets us check that our custom list is grounded in the corpus. __It also reminds us that stop-word lists are interpretive choices.__ Another researcher might keep some of these words, remove more boilerplate, or compare several preprocessing versions before choosing a final model. These layered choices are why topic modeling is usually considered an exploratory method. It is not great for supporting universal knowledge claims about culture or textual data because it ultimately involves a ton of interpretive choices.

Now we'll combine the built-in English stop words with our corpus-specific stop words, then write a small preprocessing function and deploy it to create our processed tokens, like this:

In [ ]:
# combining standard stop words with corpus-specific stop words
stop_words = set(ENGLISH_STOP_WORDS).union(UN_stopwords)

# our preprocessing steps as a function
def preprocess_text(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]{3,}", text)

    cleaned_tokens = []

    for token in tokens:
        if token not in stop_words:
            cleaned_tokens.append(token)

    return cleaned_tokens

human_rights["tokens"] = human_rights["text"].apply(preprocess_text)

We'll also need our processed text as a string in order to use `scikit-learn` and its vectorizer. So, let's join our tokens again and save the results in a "text_processed" column:

In [ ]:
def join_tokens(tokens):
    return " ".join(tokens)

human_rights["text_processed"] = human_rights["tokens"].apply(join_tokens)

human_rights.head()

To reiterate: the `tokens` column stores each cleaned document as a list of words. That is useful for `gensim`, which expects tokenized documents. The `text_processed` column stores the same words joined back into a single string. That is useful for `scikit-learn`'s vectorizer, which expects text strings.

Let's inspect the first few cleaned tokens from the first document.

In [ ]:
human_rights["tokens"][0][:10]


A second word-frequency table can help us see what remains after preprocessing. This is still not the topic model. It is a sanity check on the vocabulary we are about to model. Let's check out our most frequent words:

In [ ]:
all_tokens = []

for tokens in human_rights["tokens"]:
    for token in tokens:
        all_tokens.append(token)
        
word_counts = Counter(all_tokens)

top_words = pd.DataFrame(
    word_counts.most_common(20),
    columns=["word", "frequency"]
)

top_words

Let's plot the same post-preprocessing frequency table. If the most common words are still too generic, that usually means we should revisit the stop-word list before trusting a topic model.


In [ ]:
plt.figure(figsize=(8, 6))
plt.barh(top_words["word"], top_words["frequency"], color="steelblue")
plt.title("Most Common Words After Preprocessing")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()

## Step 3: Create A Document-Term Matrix

LDA needs word counts. To provide those counts, we'll create a __document-term matrix__, often shortened to __DTM__.

A document-term matrix is a table where:

- each row is a document
- each column is a term
- each value is the number of times that term appears in that document

This should sound familiar from the Week 3 TF-IDF tutorial. In that notebook, `TfidfVectorizer` also created a matrix with documents as rows and terms as columns. The big difference is what the numbers mean.

In a TF-IDF matrix, the values are weighted scores. A word gets a higher score when it appears often in one document but is less common across the full corpus. TF-IDF is useful when we want to find distinctive terms.

For LDA, we usually want a count matrix instead. The values in this matrix are plain word counts, because LDA models documents as mixtures of topics based on how often words appear. So the shape of the data is similar to TF-IDF, but the meaning of the numbers is different: TF-IDF stores weighted distinctiveness; this LDA matrix stores counts.

We'll use `CountVectorizer` to create this matrix. Just like `TfidfVectorizer`, it will `fit` itself to the corpus by learning the vocabulary, then `transform` each document into numbers.

In [ ]:
vectorizer = CountVectorizer(
    #ignore terms that appear in more than 90% of documents
    max_df=0.9,
    #ignore terms that appear in fewer than 2 documents
    min_df=2,
    #keep only the 500 most frequent terms after filtering
    max_features=500,
    #keep both single words and two-word phrases
    ngram_range=(1, 2),
)

dtm = vectorizer.fit_transform(human_rights["text_processed"])

print(type(dtm))
print(dtm.shape)

Let's pause on the vectorizer settings:

- `max_df=0.9` drops terms that appear in more than 90% of documents, since they may be too common to distinguish topics.
- `min_df=2` drops terms that appear in only one document, since this tiny corpus can make rare words noisy.
- `max_features=500` keeps the vocabulary small enough for a classroom example.
- `ngram_range=(1, 2)` keeps single words and two-word phrases, so terms like `death penalty` can appear as features.

The output shape tells us how many document rows and term columns the vectorizer created.

The DTM is stored as a sparse matrix, just like the TF-IDF matrix in Week 3. That means Python stores only the nonzero counts rather than printing a giant table full of zeros. This is much more efficient for text data because most documents contain only a small portion of the whole vocabulary.

We can still inspect the vocabulary that the vectorizer learned, though. The .get_feature_names_out() method makes this easy:

In [ ]:
vocab = vectorizer.get_feature_names_out()
vocab[:25]

And if we want to see the DTM in ordinary dataframe form, we can convert it. Here, the rows are files and the columns are terms from the learned vocabulary. These values are counts of the words as they appear in the documents:

In [ ]:
dtm_df = pd.DataFrame(
    #convert dtm to an array first
    dtm.toarray(),
    columns=vocab,
    index=human_rights["file"]
)

dtm_df.iloc[:, :10].head()

## Step 4: Fit The LDA Topic Model

Now we can fit the topic model. Hooray! Here, fitting means that LDA looks at the document-term matrix and estimates two things at the same time:

- which words tend to belong together in each topic
- how strongly each document is associated with each topic

The model doesn't know the correct topics in advance. We only tell it how many topics to look for, and it tries to find topic-word patterns that explain the word counts in the matrix.

The most important choice here is `n_topics`: the number of topics we ask the model to find. There is no single correct number. We'll start with five topics. Five to ten topics is a common range in DH work. Asking for too many topics in a small corpus usually creates noisy or repetitive results.

We'll also set `random_state=42` (secret of the universe) so that the model gives the same result each time we run the notebook.

In [ ]:
n_topics = 5

lda = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=30,
    learning_method="batch",
    # any number will do but 42 is the secret to the universe
    random_state=42,
)

lda.fit(dtm)

The fitted model stores topic-word weights in `lda.components_`. Each row is one topic. Each column is one term from our vectorizer vocabulary. A larger weight means that the model connects that word more strongly with that topic.

In [ ]:
print(type(lda.components_))
print(lda.components_.shape)
print(lda.components_)

Those weights are not very readable by themselves, so we'll write a small helper function that prints the most heavily weighted words for each topic.

Here's what the helper function does:

- loops through each topic row in `model.components_`
- sorts the word weights from highest to lowest
- uses those sorted positions to look up the actual vocabulary words
- returns a small dataframe with one row per topic

In [ ]:
# generated with Codex here
def get_top_words(model, feature_names, n_top_words=12):
    topic_rows = []

    for topic_idx, topic in enumerate(model.components_, start=1):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_terms = [feature_names[index] for index in top_indices]

        topic_rows.append({
            "topic": f"Topic {topic_idx}",
            "top_words": ", ".join(top_terms),
        })

    return pd.DataFrame(topic_rows)

topic_words = get_top_words(lda, vocab, n_top_words=12)
topic_words

These word lists are the starting point for interpretation. The model identified them by sorting the topic-word weights, not by reading for meaning. In other words, these are the terms the model thinks are most probable or most important for each topic.

Based on these words, can we try giving each topic a short working label? A label should summarize what the word cluster seems to be about, but it should stay tentative until you compare the topic back to actual passages in the documents.

A topic is not automatically a theme. It is a statistical cluster of terms that may or may not correspond to something interpretively meaningful.

In [ ]:
print(topic_words["top_words"][2])

We can also inspect one topic more closely. Change `topic_to_inspect` to another number from 1 to 5 and rerun the cell to see a different topic.


In [ ]:
topic_to_inspect = 5

topic_weights = pd.DataFrame({
    "word": vocab,
    "weight": lda.components_[topic_to_inspect - 1],
}).sort_values("weight", ascending=False)

topic_weights.head(15)


The `weight` column is an internal model value. The easiest way to read it is comparative: within this topic, higher-weighted words are more central to the model's idea of the topic. The exact number matters less than the ranking.

A quick bar chart makes that ranking easier to see.

In [ ]:
topic_to_plot = 5

topic_plot_df = (
    pd.DataFrame({
        "word": vocab,
        "weight": lda.components_[topic_to_plot - 1],
    })
    .sort_values("weight", ascending=False)
    .head(12)
    .sort_values("weight")
)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=topic_plot_df,
    x="weight",
    y="word",
    color="steelblue"
)
plt.title(f"Top Words for Topic {topic_to_plot}")
plt.xlabel("Topic-word weight")
plt.ylabel("Word")
plt.tight_layout()


## Step 5: Inspect Document-Topic Mixtures

LDA assumes that each document is a mixture of topics. The `.transform()` method gives us each document's topic distribution.

Each row below adds up to 1. A value near 1 means the model sees that topic as dominant in the document. A value near 0 means the topic is barely present.


In [ ]:
doc_topic = lda.transform(dtm)
topic_columns = [f"topic_{topic_num}" for topic_num in range(1, n_topics + 1)]

doc_topic_df = pd.DataFrame(doc_topic, columns=topic_columns)
doc_topic_df.insert(0, "file", human_rights["file"])
doc_topic_df["top_topic"] = doc_topic_df[topic_columns].idxmax(axis=1)

doc_topic_df


This table lets us compare documents. For example, we can ask which reports share the same dominant topic, or which reports have more mixed topic distributions.

Let's count how many documents have each topic as their highest-weighted topic.


In [ ]:
doc_topic_df["top_topic"].value_counts().sort_index()

A heatmap gives us another way to inspect the same document-topic mixtures. Darker cells indicate stronger topic weights.


In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(
    doc_topic_df.set_index("file")[topic_columns],
    cmap="YlGnBu",
    annot=True,
    fmt=".2f",
    cbar_kws={"label": "Topic weight"},
)
plt.title("Topic Mixtures by Document")
plt.xlabel("Topic")
plt.ylabel("Document")
plt.tight_layout()

## Step 6: Visualize The Model With `pyLDAvis`

`pyLDAvis` creates an interactive visualization for LDA models.

On the left, each circle represents a topic. Larger circles represent more common topics. Topics that are closer together use more similar vocabulary. On the right, the bar chart shows the most relevant terms for the selected topic.

The lambda slider changes how terms are ranked:

- Lambda near 1 emphasizes high-probability words within a topic
- Lambda near 0 emphasizes words that are more exclusive to that topic

Both views are useful. High-probability words tell you what is common in the topic. Exclusive words can help you see what makes one topic different from another.

In VS Code, wide HTML outputs sometimes get clipped. The small wrapper below adds horizontal scrolling around the visualization without changing the model code.


In [ ]:
topic_term_dists = lda.components_ / lda.components_.sum(axis=1)[:, None]
doc_topic_dists = doc_topic
doc_lengths = dtm.sum(axis=1).A1
term_frequency = dtm.sum(axis=0).A1

panel = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dists,
    doc_lengths=doc_lengths,
    vocab=vocab,
    term_frequency=term_frequency,
    mds="pcoa",
    sort_topics=False,
)

pyLDAvis.display(panel)

### Reading The Visualization Carefully

When you use this visualization, try asking:

- Which topics overlap? What shared language might explain that overlap?
- Which terms are common but not very distinctive?
- Which terms become more important when you lower lambda?
- Do the document-topic mixtures make sense after you read excerpts from the documents?

If the topics are too repetitive, try changing the preprocessing, the number of topics, or the vectorizer settings.

## Step 7: Run A Comparable Model With `gensim`

So far, our main model used `scikit-learn`. Another common topic-modeling library is `gensim`.

The conceptual workflow is similar, but the data structures are a little different:

- `corpora.Dictionary()` learns the vocabulary from tokenized texts
- `.doc2bow()` turns each tokenized document into word-count pairs
- `LdaModel()` fits the topic model from those word-count pairs

`bow` stands for "bag of words." A bag-of-words representation ignores word order and keeps counts. That is very similar in spirit to the document-term matrix we created for `scikit-learn`, but `gensim` stores each document as a compact list of `(word_id, count)` pairs instead of one large matrix.

Because we already created a `tokens` column, we can reuse it here. Let's put that all together:

In [ ]:
dictionary = corpora.Dictionary(human_rights["tokens"])
dictionary.filter_extremes(no_below=2, no_above=0.9, keep_n=500)
dictionary.compactify()

corpus = [dictionary.doc2bow(tokens) for tokens in human_rights["tokens"]]

print(f"Dictionary terms: {len(dictionary)}")
print(f"Documents in corpus: {len(corpus)}")
print(corpus[0][:10])

Each item in `corpus` is a document represented as word IDs and counts. For example, `(17, 3)` would mean that the word with ID 17 appears 3 times in that document.

Now we'll fit a `gensim` LDA model with the same number of topics. The main fitting choices below are `passes`, which controls how many times the model works through the whole corpus, and `iterations`, which controls how much work it does inside each pass. More passes and iterations can make the model more stable, but they also take longer.


In [ ]:
lda_gensim = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=n_topics,
    random_state=42,
    passes=30,
    iterations=100,
)

for topic_id, terms in lda_gensim.show_topics(num_topics=n_topics, num_words=12, formatted=False):
    words = ", ".join(word for word, weight in terms)
    print(f"Topic {topic_id + 1}: {words}")

The `gensim` topics will not match the `scikit-learn` topics exactly. That is normal. The packages use different implementations, and LDA itself is probabilistic.

Instead of looking for identical output, compare the broader patterns. Which word clusters seem stable across both models? Which ones change?

We can also ask `gensim` for each document's topic mixture, just as we did with the `scikit-learn` model.


In [ ]:
gensim_doc_topics = []

for file_name, bow in zip(human_rights["file"], corpus):
    topic_distribution = lda_gensim.get_document_topics(bow, minimum_probability=0)
    row = {"file": file_name}

    for topic_id, weight in topic_distribution:
        row[f"topic_{topic_id + 1}"] = weight

    gensim_doc_topics.append(row)

gensim_doc_topic_df = pd.DataFrame(gensim_doc_topics)
gensim_doc_topic_df["top_topic"] = gensim_doc_topic_df[topic_columns].idxmax(axis=1)

gensim_doc_topic_df


Here's a quick visualization of the `gensim` document-topic mixtures. This is the same kind of heatmap we used for the `scikit-learn` model. How do they compare?

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(
    gensim_doc_topic_df.set_index("file")[topic_columns],
    cmap="YlOrRd",
    annot=True,
    fmt=".2f",
    cbar_kws={"label": "Topic weight"},
)
plt.title("Gensim Topic Mixtures by Document")
plt.xlabel("Topic")
plt.ylabel("Document")
plt.tight_layout()